# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hariommishra-12/Flyrank-Project/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:

!pip install -q -U duckdb huggingface_hub scikit-learn

import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("SET hf_token = getenv('HF_TOKEN');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
print("connected")

## 1. Question

**Research question:** Given a content item's search performance and freshness signals over a
3-month window, can we rank which pages are likely to decline in the following 2 months —
well enough that an editor reviewing the top of the ranked list catches real declines more
often than reviewing pages in any order?

**Decision it supports:** which pages a content editor opens first this week, given limited
review hours.

In [ ]:
# No query yet -- this section is the framing, carried over from ML-02/ML-03.
print("Lane: Refresh / Content Opportunity Scoring")
print("Unit of analysis: one (client_hash_id, content_hash_id) pair, aggregated monthly")

## 2. Data

**Release:** FlyRank Internship Warehouse, v20260703 (frozen snapshot, export date 2026-07-03).

**Tables used:** `fact_content_daily_performance` (partitioned by month), `dim_clients`,
`dim_content`.

**Windows:**
- Feature window (dev): Dec 2025 – Feb 2026 (3 months, mid-panel, never the sealed month)
- Label window (dev): Mar – Apr 2026 (strictly after the feature window)
- Sealed test feature window: Mar – May 2026
- Sealed test label window: June 2026 (`_sample` — touched exactly once, at the end)

**Excluded:** clients with `access_profile = 'no_search_or_analytics_access'` (no signal to
rank on); rows where `ga4_data_available = FALSE` are kept for GSC-only features but excluded
from any engagement feature. [Add real row/client counts here once you've run the query below.]

In [ ]:
# Confirm feature/label windows exist and get real counts to cite in the markdown above.
feat_months = ["2025-12", "2026-01", "2026-02"]
label_months = ["2026-03", "2026-04"]

def month_union(months):
    parts = [f"read_parquet('{BASE}/fact_content_daily_performance/month={m}/*.parquet')" for m in months]
    return " UNION ALL ".join(f"SELECT * FROM {p}" for p in parts)

con.sql(f"""
    SELECT COUNT(*) n_rows, COUNT(DISTINCT client_hash_id) n_clients,
           COUNT(DISTINCT content_hash_id) n_content, MIN(report_date) min_d, MAX(report_date) max_d
    FROM ({month_union(feat_months)})
""").show()

con.sql(f"""
    SELECT COUNT(*) n_rows, COUNT(DISTINCT client_hash_id) n_clients
    FROM ({month_union(label_months)})
""").show()

# How many clients does the "excluded" rule above actually affect?
con.sql(f"""
    SELECT access_profile, COUNT(*) n_clients
    FROM read_parquet('{BASE}/dim_clients/*.parquet')
    GROUP BY 1 ORDER BY 2 DESC
""").show()

## 3. Methodology

**Features** (all from the feature window only, never touching the label window):
avg(gsc_avg_position), sum(gsc_impressions), sum(gsc_clicks), ctr, sum(sessions_ai), content
age at start of feature window, days since last update, word count, ga4 engagement rate (where
`ga4_data_available = TRUE`).

**Label:** `declined = 1` if total impressions in the label window (Mar–Apr) fell more than 20%
versus the monthly-average pace of the feature window (Dec–Feb) — a genuinely future-observed
outcome, not a same-window rule.

**Baseline:** a transparent rule — rank by feature-window impressions × (1 / recency of last
update) — the same style of hand-built score as the starter pipeline's baseline.

**Validation:** GroupKFold by `client_hash_id` (never a random row split — clients repeat, and
a random split would let the model memorize client identity). I report both the random-split
and grouped-split precision@50 and treat the gap between them as a leakage signal, per
`skills/hunting-leakage-and-validating`.

**Leakage checks:** the label's own ingredient (impressions in Mar–Apr) is never a feature;
every feature is drawn strictly from Dec–Feb; population is all content active in the feature
window, not filtered by anything from the label window.

In [ ]:
import pandas as pd

# Build the feature table (feature window only).
features_sql = f"""
    SELECT
        client_hash_id, content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS impressions_feat,
        SUM(gsc_clicks) AS clicks_feat,
        SUM(sessions_ai) AS ai_sessions_feat
    FROM ({month_union(feat_months)})
    GROUP BY 1, 2
"""
features_df = con.sql(features_sql).df()

# Build the label table (label window only) -- strictly later dates.
label_sql = f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_label
    FROM ({month_union(label_months)})
    GROUP BY 1, 2
"""
label_df = con.sql(label_sql).df()

df = features_df.merge(label_df, on=["client_hash_id", "content_hash_id"], how="inner")
monthly_avg_feat = df["impressions_feat"] / len(feat_months)
monthly_avg_label = df["impressions_label"] / len(label_months)
df["declined"] = (monthly_avg_label < monthly_avg_feat * 0.8).astype(int)

print("rows:", len(df), " base rate (declined):", round(df["declined"].mean(), 3))
df.head()

## 4. Results (vs baseline)

[Fill in after running the code below.] Report: baseline precision@50, model precision@50 on
random split, model precision@50 on grouped split, and the base rate. State the gap between
random and grouped splits explicitly — if it's large, say so as a finding, not a footnote.

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score

feature_cols = ["avg_position", "impressions_feat", "clicks_feat", "ai_sessions_feat"]
X, y, groups = df[feature_cols].fillna(0), df["declined"], df["client_hash_id"]

def precision_at_k(y_true, scores, k=50):
    top_k_idx = scores.sort_values(ascending=False).index[:k]
    return y_true.loc[top_k_idx].mean()

# Baseline: rank by raw impressions_feat (a simple, transparent rule).
baseline_p50 = precision_at_k(y, df["impressions_feat"], k=50)

# Random split (the number to compare against, to expose memorization).
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_tr, y_tr)
scores_random = pd.Series(rf.predict_proba(X_te)[:, 1], index=X_te.index)
random_split_p50 = precision_at_k(y_te, scores_random, k=50)

# Grouped split (the honest number).
gkf = GroupKFold(n_splits=5)
tr_idx, te_idx = next(gkf.split(X, y, groups))
rf2 = RandomForestClassifier(n_estimators=200, random_state=0).fit(X.iloc[tr_idx], y.iloc[tr_idx])
scores_grouped = pd.Series(rf2.predict_proba(X.iloc[te_idx])[:, 1], index=X.iloc[te_idx].index)
grouped_split_p50 = precision_at_k(y.iloc[te_idx], scores_grouped, k=50)

print(f"base rate:            {y.mean():.3f}")
print(f"baseline precision@50: {baseline_p50:.3f}")
print(f"random-split  p@50:    {random_split_p50:.3f}")
print(f"grouped-split p@50:    {grouped_split_p50:.3f}")
print(f"random vs grouped gap: {random_split_p50 - grouped_split_p50:.3f}")

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
